# M63 — R0 → A2 Vehicle-depth distillation pilot
**Revision M63-2026-09-28-r1. Run sections 1–10 in order. Select NVIDIA L4, matching M62. No phone needed.**
Two 10-epoch runs from A2 epoch130: GT-only control and GT + fixed 0.25 metre-L1 depth teaching. R0 epoch185 supplies only the 7,014 reviewed Vehicle depth targets. Both classes retain all original GT losses. MonoDGP/M61 remain parked.
This is an experiment, not a guaranteed accuracy improvement. No longer training or deployment is automatically authorized.
Restart: rerun sections 1 onward. Preparation verifies existing targets; training resumes from the last complete epoch. Keep GPU/software unchanged. A failed mid-epoch is replayed. Never delete provenance to bypass a failure.


## 1. Drive, paths, and logging


In [ ]:
# M63-2026-09-28-r1 — visible notebook revision.
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
from datetime import datetime, timezone
import json, os, shlex, shutil, subprocess, sys, zipfile

print('M63-2026-09-28-r1')
DRIVE = Path('/content/drive/MyDrive')
MOBILE_REPO = Path('/content/mobile_adas3d')
REPO = Path('/content/MonoDETR_M62')
COMMIT = '6994b9f512400b258c6edb75f77423beb9c126f2'
OUTPUT = DRIVE / 'mobile_adas3d_outputs/students/monodetr_m62_r0_a2_diagnostic'
M62_OUTPUT = OUTPUT
OUTPUT = DRIVE / 'mobile_adas3d_outputs/students/monodetr_m63_r0_a2_depth_pilot'
MANIFEST = OUTPUT / 'm63_manifest.json'
DATASET = Path('/content/kitti_m62')
SPLITS = DRIVE / 'mobile_adas3d_splits/kitti_chen'
R0_SELECTION = DRIVE / 'mobile_adas3d_outputs/references/monodetr_r0/product_checkpoint_sweep/r0_product_selection.json'
A2_SELECTION = DRIVE / 'mobile_adas3d_outputs/students/monodetr_a2_gt/product_checkpoint_sweep/a2_product_selection.json'
DATASET_CANDIDATES = [DATASET, Path('/content/kitti_m61'), Path('/content/kitti'),
                      Path('/content/monodetr_kitti'), DRIVE / 'datasets/kitti']

def run(command, name, cwd=None):
    command = [str(x) for x in command]
    log_path = OUTPUT / 'colab_logs' / (name + '.log')
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('+', shlex.join(command), '\nDurable log:', log_path, flush=True)
    tail = deque(maxlen=80)
    env = dict(os.environ, PYTHONUNBUFFERED='1')
    with log_path.open('a', buffering=1) as log:
        log.write('\nSTART ' + datetime.now(timezone.utc).isoformat() + '\n')
        p = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            print(line, end='', flush=True)
            log.write(line)
            tail.append(line.rstrip())
        rc = p.wait()
    if rc:
        raise RuntimeError(f'Exit {rc}; log={log_path}\n' + '\n'.join(tail))

def pilot(script, *args):
    run([sys.executable, '-u', MOBILE_REPO / 'scripts' / script,
         '--manifest', MANIFEST, *args], script + '_' + '_'.join(args), MOBILE_REPO)

run(['nvidia-smi'], 'gpu')
M62 = json.loads((M62_OUTPUT / 'm62_manifest.json').read_text())
if M62['repo'] != str(REPO) or M62['dataset_root'] != str(DATASET):
    raise RuntimeError('Restore the original M62 paths; do not edit the frozen manifest.')


## 2. Restore pinned MonoDETR and the M62 environment
Use main. No reset or deletion of existing source. The exact M62 torch/CUDA, GPU and scientific package versions are required to reuse its caches. If the runtime changed, stop and report the environment difference.


In [ ]:
mobile_# cv2.__version__ is not necessarily the distribution version (wheel suffix).
# Validate the actual imported versions before rebuilding the native extension.
probe = ("import sys; sys.path.insert(0, sys.argv[1]); "
         "from m61_common import environment; from diagnose_m62_r0_a2 import require_identity; "
         "import json; require_identity(json.load(open(sys.argv[2]))['environment'], environment())")
run([sys.executable, '-c', probe, MOBILE_REPO / 'scripts', M62_OUTPUT / 'm62_manifest.json'],
    'verify_m62_environment', MOBILE_REPO)
url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE_REPO.exists():
    run(['git', 'clone', '--branch', 'main', mobile_url, MOBILE_REPO], 'clone_mobile')
branch = subprocess.check_output(['git', 'branch', '--show-current'], cwd=MOBILE_REPO, text=True).strip()
if branch != 'main':
    raise RuntimeError(f'Use main, not {branch!r}')
run(['git', 'pull', '--ff-only'], 'update_mobile', MOBILE_REPO)
deps = M62['environment']
requirements = ['pyyaml', 'numba', 'scikit-image', 'scikit-learn', 'tqdm', 'ninja', 'pandas',
                'timm==' + deps['timm'], 'numpy==' + deps['numpy'],
                'scipy==' + deps['scipy'], 'pillow==' + deps['pillow'],
                'opencv-python-headless']
run([sys.executable, '-m', 'pip', 'install', '-q', *requirements], 'dependencies')
url = '/'.join(['https:', '', 'github.com', 'ZrrSkywalker', 'MonoDETR.git'])
if not REPO.exists():
    run(['git', 'clone', url, REPO], 'clone_monodetr')
head = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
if head != COMMIT:
    if subprocess.check_output(['git', 'status', '--porcelain'], cwd=REPO, text=True).strip():
        raise RuntimeError('Wrong upstream commit with local changes; use a new dedicated directory.')
    run(['git', 'fetch', 'origin', COMMIT], 'fetch_monodetr', REPO)
    run(['git', 'checkout', '--detach', COMMIT], 'pin_monodetr', REPO)
for patch in ('patch_monodetr_colab_compat.py', 'patch_monodetr_product_taxonomy.py',
              'patch_monodetr_mobilenetv4.py'):
    run([sys.executable, MOBILE_REPO / 'scripts' / patch, '--monodetr-repo', REPO], patch, MOBILE_REPO)
ops = REPO / 'lib/models/monodetr/ops'
patch_arch = ("import sys; from pathlib import Path; sys.path.insert(0, sys.argv[1]); "
              "from setup_m61_sources import patch_active_architecture; "
              "patch_active_architecture(Path(sys.argv[2]))")
run([sys.executable, '-c', patch_arch, MOBILE_REPO / 'scripts', ops / 'setup.py'], 'cuda_arch', MOBILE_REPO)
os.environ['MAX_JOBS'] = '2'
run([sys.executable, 'setup.py', 'build_ext', '--inplace'], 'cuda_build', ops)


## 3. Restore KITTI and both exact Chen splits
All 3,712 training and 3,769 validation samples are needed. M62 cache inputs are checked against the training view during both runs.


In [ ]:
(DATASET / 'training').mkdir(parents=True, exist_ok=True)
(DATASET / 'ImageSets').mkdir(parents=True, exist_ok=True)
for name in ('image_2', 'label_2', 'calib'):
    alternatives = [f'training/{name}']
    if name in ('image_2', 'label_2'):
        alternatives.append(f'training/{name[:-1]}02')
    source = next((root / relative for root in DATASET_CANDIDATES for relative in alternatives
                   if (root / relative).is_dir()), None)
    if source is None:
        raise FileNotFoundError(f'Missing {name}; edit DATASET_CANDIDATES in section1.')
    link = DATASET / 'training' / name
    if not link.exists() and not link.is_symlink():
        link.symlink_to(source.resolve(), target_is_directory=True)
    elif link.resolve() != source.resolve():
        raise RuntimeError(f'Existing dataset link differs: {link}')
for split in ('train', 'val'):
    src, dst = SPLITS / (split + '.txt'), DATASET / 'ImageSets' / (split + '.txt')
    if dst.exists() and dst.read_bytes() != src.read_bytes():
        raise RuntimeError(f'Split changed: {dst}')
    shutil.copy2(src, dst)


## 4. Tests and target preparation
Verifies the reviewed M62 reports, complete cache hashes, environment and all target files. Recomputes exactly 7,014 approved depth objects. Existing M62 evidence is read-only. This may take time on Drive; progress prints every 100 images.


In [ ]:
run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests',
     '-p', 'test_m63_depth_pilot.py', '-v'], 'tests', MOBILE_REPO)
run([sys.executable, '-u', MOBILE_REPO / 'scripts/prepare_m63_depth_pilot.py',
     '--m62-output', M62_OUTPUT, '--output', OUTPUT], 'prepare', MOBILE_REPO)


## 5. Fresh full-validation A2 baseline
Before training, reproduce A2 within 0.15 AP points and 0.01 nearby recall of its recorded metrics. This is a protocol-drift check, not a revised product target. Stop for review on failure.


In [ ]:
pilot('evaluate_m63_pilot.py', '--baseline-only')


## 6. Real CUDA gradient smoke — zero optimizer steps
Requires finite GT/KD loss and gradients, nonzero teacher gradient, and an empty-GT check when present.


In [ ]:
pilot('train_m63_student.py', '--smoke')


## 7. Control — 10 epochs, GT only
Fresh AdamW, learning rate 1e-5, batch 4, seed 20268, FP32, no augmentation. Saves an atomic checkpoint every epoch on Drive.


In [ ]:
pilot('train_m63_student.py', '--variant', 'control')


## 8. Treatment — 10 epochs, GT + Vehicle depth KD
Starts independently from the same A2 epoch130. Same order, optimizer and GT losses. Adds 0.25 × mean absolute depth difference in metres on approved GT-matched Vehicle queries. No temperature applies to this regression loss.


In [ ]:
pilot('train_m63_student.py', '--variant', 'vehicle_kd')


## 9. Compare epoch 10 on the complete validation split
No checkpoint sweep. Require Vehicle 3D gain ≥0.10 AP over both baseline and control; preserve Pedestrian AP/recall and balanced 3D AP against both. Allow at most 0.15 Vehicle BEV AP and 0.01 Vehicle nearby recall loss. Historical product gates remain separately reported. A pilot pass only recommends confirmation review.


In [ ]:
pilot('evaluate_m63_pilot.py')


## 10. Download results — STOP for review
Return the ZIP before any longer training, compression, or iPhone work. A failed scientific gate is still a valid result; do not tune the fixed criteria afterwards.


In [ ]:
bundle = OUTPUT / 'm63_results.zip'
names = ['m63_manifest.json', 'm63_teacher_audit.json', 'm63_baseline_metrics.json',
         'm63_training_smoke.json', 'm63_pilot_comparison.json', 'm63_pilot_comparison.csv',
         'runs/control/training_summary.json', 'runs/vehicle_kd/training_summary.json']
with zipfile.ZipFile(bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for name in names:
        path = OUTPUT / name
        if not path.is_file(): raise FileNotFoundError(path)
        z.write(path, name)
print('Return:', bundle, '\\nSTOP for review; no deployment authorization.')
from google.colab import files
files.download(str(bundle))
